In [7]:
!pip -q install pyspark openpyxl requests

In [9]:
import pandas as pd
import sqlite3
import json
import requests
from pathlib import Path


df = pd.read_excel("week12_transactions.xlsx")


if pd.api.types.is_numeric_dtype(df["date"]):
    df["date"] = pd.to_datetime(df["date"], unit="D", origin="1899-12-30")
else:
    df["date"] = pd.to_datetime(df["date"])

df["date"] = df["date"].dt.strftime("%Y-%m-%d")
df.to_csv("week12_transactions.csv", index=False)

print("Rows:", len(df))
print("Total value (RWF):", f"{df['amount_rwf'].sum():,}")
display(df.head())


Rows: 2000
Total value (RWF): 119,876,000


,txn_id,date,hour,district,txn_type,amount_rwf,flagged
0,T00001,2026-11-01,8,Gasabo,send,1000,0
1,T00002,2026-11-09,12,Nyarugenge,send,25000,0
2,T00003,2026-11-19,8,Muhanga,send,51000,0
3,T00004,2026-11-01,17,Huye,send,79000,0
4,T00005,2026-11-15,12,Huye,send,109000,0


Exercise 1.1 — SELECT / WHERE / ORDER BY

In [11]:
conn = sqlite3.connect("momo.db")
df.to_sql("transactions", conn, if_exists="replace", index=False)

# (a) Five largest SEND transactions above 100,000 RWF.
q_1a = '''
SELECT txn_id, district, amount_rwf
FROM transactions
WHERE amount_rwf > 100000 AND txn_type = 'send'
ORDER BY amount_rwf DESC
LIMIT 5;
'''
top_five_sends = pd.read_sql(q_1a, conn)
display(top_five_sends)

# (b) Count flagged transactions.
q_1b = '''
SELECT COUNT(*) AS flagged_count
FROM transactions
WHERE flagged = 1;
'''
display(pd.read_sql(q_1b, conn))


,txn_id,district,amount_rwf
0,T01708,Kicukiro,140000
1,T00364,Huye,139000
2,T00895,Muhanga,139000
3,T00899,Nyarugenge,139000
4,T01314,Gasabo,139000


,flagged_count
0,11


Exercise 1.2 — GROUP BY

In [12]:
# (a) Count, sum and average amount for each transaction type.
q_1_2a = '''
SELECT txn_type,
       COUNT(*) AS n_txns,
       SUM(amount_rwf) AS total_rwf,
       AVG(amount_rwf) AS avg_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_rwf DESC;
'''
type_summary = pd.read_sql(q_1_2a, conn)
display(type_summary)
# Answer: send moves the most money.

# (b) Total transaction value per district.
q_1_2b = '''
SELECT district, SUM(amount_rwf) AS total_rwf
FROM transactions
GROUP BY district
ORDER BY total_rwf DESC;
'''
district_summary = pd.read_sql(q_1_2b, conn)
display(district_summary)


,txn_type,n_txns,total_rwf,avg_rwf
0,send,868,62049000,71485.023041
1,paybill,330,29970000,90818.181818
2,receive,489,27024000,55263.803681
3,airtime,313,833000,2661.341853


,district,total_rwf
0,Muhanga,21076000
1,Gasabo,20738000
2,Huye,18736000
3,Nyarugenge,18091000
4,Kicukiro,11338000
5,Musanze,10490500
6,Rubavu,10110500
7,Nyagatare,9296000


Exercise 1.3 — The busiest hour

In [13]:
# (a) SQL: transactions per hour, busiest first.
q_1_3 = '''
SELECT hour, COUNT(*) AS n_txns
FROM transactions
GROUP BY hour
ORDER BY n_txns DESC, hour;
'''
sql_hour_summary = pd.read_sql(q_1_3, conn)
display(sql_hour_summary)

# (b) Repeat in Pandas and verify that the two results agree.
pandas_hour_summary = (
    df.groupby("hour").size().reset_index(name="n_txns")
      .sort_values(["n_txns", "hour"], ascending=[False, True])
      .reset_index(drop=True)
)
display(pandas_hour_summary)
assert sql_hour_summary.equals(pandas_hour_summary)
print("SQL and Pandas agree.")
print("Busiest hour:", int(sql_hour_summary.iloc[0]["hour"]),
      "with", int(sql_hour_summary.iloc[0]["n_txns"]), "transactions.")


,hour,n_txns
0,20,185
1,17,183
2,18,176
3,12,170
4,11,169
5,8,168
6,10,168
7,19,158
8,9,144
9,16,38


,hour,n_txns
0,20,185
1,17,183
2,18,176
3,12,170
4,11,169
5,8,168
6,10,168
7,19,158
8,9,144
9,16,38


SQL and Pandas agree.
Busiest hour: 20 with 185 transactions.


Part 2 · APIs & JSON

In [14]:
raw = '{"city": "Kigali", "temps": [21.5, 24.0, 26.2]}'
data = json.loads(raw)

print("City:", data["city"])
print("Number of readings:", len(data["temps"]))
print(f"Average temperature: {sum(data['temps']) / len(data['temps']):.2f} °C")


City: Kigali
Number of readings: 3
Average temperature: 23.90 °C


Exercise 2.2 — Call a real API

In [15]:
url = (
    "https://api.open-meteo.com/v1/forecast"
    "?latitude=-1.95&longitude=30.06"
    "&hourly=temperature_2m&forecast_days=1"
)

try:
    response = requests.get(url, timeout=15)
    response.raise_for_status()
    api_data = response.json()
    times = api_data["hourly"]["time"]
    temps = api_data["hourly"]["temperature_2m"]
    source = "Live Open-Meteo API"
except Exception as api_error:
    print("Live API request failed:", api_error)
    sample_path = Path("week12_api_sample.json")
    if not sample_path.exists():
        raise FileNotFoundError(
            "Upload the instructor-provided week12_api_sample.json to run "
            "the offline fallback. No sample file was included with the "
            "uploaded lab materials, so no forecast values are fabricated."
        )
    api_data = json.loads(sample_path.read_text())
    # Expected sample structure follows Open-Meteo's hourly object.
    times = api_data["hourly"]["time"]
    temps = api_data["hourly"]["temperature_2m"]
    source = "Instructor offline sample"

print("Source:", source)
print("Number of hours:", len(temps))
print("Maximum temperature:", max(temps), "°C")
print("Minimum temperature:", min(temps), "°C")
hottest_index = temps.index(max(temps))
print("Time of hottest hour:", times[hottest_index])


Source: Live Open-Meteo API
Number of hours: 24
Maximum temperature: 26.6 °C
Minimum temperature: 15.4 °C
Time of hottest hour: 2026-11-26T11:00


Exercise 2.3 — API data meets your toolbox

In [16]:
weather_df = pd.DataFrame({"time": times, "temp_c": temps})
display(weather_df.head())
display(weather_df["temp_c"].describe())

# Count readings at or above 25 degrees Celsius.
hours_at_least_25 = (weather_df["temp_c"] >= 25).sum()
print("Hours >= 25 °C:", int(hours_at_least_25))


,time,temp_c
0,2026-11-26T00:00,17.2
1,2026-11-26T01:00,16.3
2,2026-11-26T02:00,15.9
3,2026-11-26T03:00,15.5
4,2026-11-26T04:00,15.4


count    24.000000
mean     20.016667
std       3.521569
min      15.400000
25%      17.025000
50%      19.450000
75%      22.400000
max      26.600000
Name: temp_c, dtype: float64

Hours >= 25 °C: 3


Part 3 · PySpark

Exercise 3.1 — Load & inspect

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum

spark = (SparkSession.builder
         .appName("AUCA_Week12_MobileMoney")
         .getOrCreate())

sdf = spark.read.csv(
    "week12_transactions.csv",
    header=True,
    inferSchema=True
)

print("Row count:", sdf.count())
sdf.printSchema()
sdf.show(5, truncate=False)


C:\Users\HONORE\anaconda3\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Row count: 2000
root
 |-- txn_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- district: string (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- amount_rwf: integer (nullable = true)
 |-- flagged: integer (nullable = true)

+------+----------+----+----------+--------+----------+-------+
|txn_id|date      |hour|district  |txn_type|amount_rwf|flagged|
+------+----------+----+----------+--------+----------+-------+
|T00001|2026-11-01|8   |Gasabo    |send    |1000      |0      |
|T00002|2026-11-09|12  |Nyarugenge|send    |25000     |0      |
|T00003|2026-11-19|8   |Muhanga   |send    |51000     |0      |
|T00004|2026-11-01|17  |Huye      |send    |79000     |0      |
|T00005|2026-11-15|12  |Huye      |send    |109000    |0      |
+------+----------+----+----------+--------+----------+-------+
only showing top 5 rows


Exercise 3.2 — Filter

In [18]:
# (a) Count transactions above 100,000 RWF.
large_transactions = sdf.filter(col("amount_rwf") > 100000)
print("Transactions above 100,000 RWF:", large_transactions.count())

# (b) Show five flagged transactions.
sdf.filter(col("flagged") == 1).show(5, truncate=False)


Transactions above 100,000 RWF: 451
+------+----------+----+--------+--------+----------+-------+
|txn_id|date      |hour|district|txn_type|amount_rwf|flagged|
+------+----------+----+--------+--------+----------+-------+
|T00279|2026-11-29|19  |Muhanga |send    |127000    |1      |
|T00326|2026-11-07|22  |Huye    |send    |127000    |1      |
|T00536|2026-11-27|17  |Gasabo  |send    |127000    |1      |
|T00655|2026-11-30|17  |Rubavu  |send    |134000    |1      |
|T00697|2026-11-01|9   |Muhanga |send    |127000    |1      |
+------+----------+----+--------+--------+----------+-------+
only showing top 5 rows


Exercise 3.3 — groupBy

In [19]:
# (a) Total amount per transaction type.
sdf.groupBy("txn_type").agg(
    spark_sum("amount_rwf").alias("total_rwf")
).orderBy(col("total_rwf").desc()).show()

# (b) Per-district totals, converted to Pandas and sorted.
spark_district_totals = (
    sdf.groupBy("district")
       .agg(spark_sum("amount_rwf").alias("total_rwf"))
       .toPandas()
       .sort_values("total_rwf", ascending=False)
)
display(spark_district_totals)

# (c) Comparison: Spark totals should agree with Part 1 SQL totals.


+--------+---------+
|txn_type|total_rwf|
+--------+---------+
|    send| 62049000|
| paybill| 29970000|
| receive| 27024000|
| airtime|   833000|
+--------+---------+



C:\Users\HONORE\anaconda3\Lib\site-packages\pyspark\sql\pandas\conversion.py:324: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,district,total_rwf
4,Muhanga,21076000
5,Gasabo,20738000
6,Huye,18736000
1,Nyarugenge,18091000
0,Kicukiro,11338000
2,Musanze,10490500
7,Rubavu,10110500
3,Nyagatare,9296000


Bonus (+3) — Spark speaks SQL too

In [20]:
sdf.createOrReplaceTempView("transactions")

spark_sql_result = spark.sql('''
SELECT txn_type,
       COUNT(*) AS n_txns,
       SUM(amount_rwf) AS total_rwf,
       AVG(amount_rwf) AS avg_rwf
FROM transactions
GROUP BY txn_type
ORDER BY total_rwf DESC
''')
spark_sql_result.show()


+--------+------+---------+------------------+
|txn_type|n_txns|total_rwf|           avg_rwf|
+--------+------+---------+------------------+
|    send|   868| 62049000| 71485.02304147465|
| paybill|   330| 29970000| 90818.18181818182|
| receive|   489| 27024000|  55263.8036809816|
| airtime|   313|   833000|2661.3418530351437|
+--------+------+---------+------------------+



Part 4 · Choose Your Weapon

1.SQL — The Ministry's data is already in PostgreSQL, so SQL can calculate monthly totals per district directly in the database.

2.Pandas — 40,000 rows are manageable on a laptop, and Pandas is suitable for cleaning, chart preparation and regression-model workflows.

3.Spark — 40 TB already on a cluster requires distributed processing; Spark can spread the work across the cluster.

4.API — A weather API can fetch today's weather automatically whenever the dashboard refreshes, without requiring a file upload.

Part 5 · Final-Project Proposal

Title: Understanding Mobile-Money Transaction Patterns and Flagged Transfers

Our project will answer the question: What patterns in mobile-money transactions are associated with transactions being flagged? We will use the supplied mobile-money transaction dataset, containing 2,000 transactions from November 2026, with transaction date, hour, district, transaction type, amount and a flagged indicator. This matters because understanding transaction patterns can help service providers monitor unusual activity and improve customer protection. We plan to build a classification model to predict whether a transaction is flagged, because the target variable (flagged) has two classes (0 and 1). We will clean the data, perform exploratory analysis, visualize transaction amounts and patterns by district, hour and type, and evaluate the model using precision, recall and F1-score. The dataset contains only 11 flagged transactions, so the class imbalance will be reported and addressed carefully; model performance will not be overstated.